# Stage 3: Neural Network with Adam Optimization

This stage uses Adam-style optimization to train the same dense network with mini-batches and adaptive parameter updates.

Adam keeps two running summaries of each gradient: the first moment estimates the recent mean direction of the gradient, and the second raw moment estimates the recent mean squared gradient size. That lets the optimizer keep momentum while scaling steps down for parameters with large or noisy gradients.

Import NumPy for array operations, manual matrix math, random mini-batch shuffling, and dataset loading.


In [1]:
import numpy as np

Load the saved dataset from the shared archive used by the earlier stages.


In [2]:
loaded_data = np.load("data/gopher_dataset.npz")

Extract the training and test images plus their binary labels from the dataset archive.


In [3]:
X_train = loaded_data["X_train"]
y_train = loaded_data["y_train"]
X_test = loaded_data["X_test"]
y_test = loaded_data["y_test"]

Inspect the raw training-image shape before flattening. The first dimension is the number of examples `m`; later mini-batches slice this axis into smaller gradient estimates.


In [4]:
X_train.shape

(44, 128, 128, 3)

Flatten and normalize the images. Dense layers multiply matrices by feature vectors, so each image becomes one vector; scaling pixels to `[0, 1]` keeps activations and gradients from being dominated by raw 0-255 units.

Labels are reshaped to `(m, 1)` so each sigmoid probability has one matching binary target.


In [5]:
X_flat = X_train.reshape(X_train.shape[0], -1) / 255.0
X_test = X_test.reshape(X_test.shape[0], -1) / 255.0
y_train = y_train.reshape(y_train.shape[0], 1)
y_test = y_test.reshape(y_test.shape[0], 1)
X_flat.shape

(44, 49152)

Initialize parameters for each network layer. Small random `W` values break symmetry between hidden units, and zero `B` values give each layer a neutral starting offset.

Adam will adapt update sizes later, but it still needs non-identical hidden weights so different neurons can learn different features.


In [7]:
def initialize_parameters(layer_dims):
    
    W = [np.random.randn(layer_dims[i], layer_dims[i-1]) * 0.01 for i in range(1, len(layer_dims))]
    B = [np.zeros((layer_dims[i], 1)) for i in range(1, len(layer_dims))]

    return W, B

Define the network architecture and create the first set of weights and biases. The dimensions encode the matrix contract: input features flow through 64 hidden units, then 32 hidden units, then one sigmoid output.


In [ ]:
layer_dims = [X_flat.shape[1], 64, 32, 1]
W, B = initialize_parameters(layer_dims)

Define the forward pass for all layers. Each layer applies `Z[l] = W[l] A[l-1] + b[l]`; hidden layers use ReLU and the final layer uses sigmoid.

The cached `(A, Z)` values are useful because backprop needs both the previous activations for `dW` and the pre-activation `Z` to apply the ReLU derivative.


In [9]:
def forward_pass(X, W, B, layer_dims):
    cache = {}
    A = X.T
    for i in range(len(layer_dims) - 1):
        print(f"Layer {i+1}: W shape {W[i].shape}, B shape {B[i].shape}, A shape {A.shape}")
        Z = np.dot(W[i], A) + B[i]
        if i == len(layer_dims) - 2:  # Output layer
            A = 1 / (1 + np.exp(-Z))  # Sigmoid activation
        else:  # Hidden layers
            A = np.maximum(0, Z)  # ReLU activation
        cache[i] = (A, Z)

    return A, cache

Define one mini-batch Adam update. Backprop first computes a gradient `g` for each parameter, exactly like the previous notebook.

Adam then updates two exponential moving averages: `v = B1 * v + (1 - B1) * g` is the first moment, a smoothed estimate of gradient direction; `s = B2 * s + (1 - B2) * g^2` is the second raw moment, a smoothed estimate of squared gradient size. The parameter step uses `v / sqrt(s + epsilon)`, so consistent directions build momentum while large or noisy gradients get smaller effective steps.

This notebook keeps the moving averages directly as a compact teaching version; full Adam commonly adds bias correction during the earliest updates.


In [ ]:
def gradient_descent(X, Y, W, B, Vdw, Sdw, Vdb, Sdb, layer_dims, learning_rate = 0.01, B1 = 0.9, B2 = 0.999, epsilon = 1e-8):
    y_pred, cache = forward_pass(X, W, B, layer_dims)
    epsilon = 1e-7
    y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon) # Avoid log(0)
    cost = -np.mean(Y.T * np.log(y_pred_clipped) + (1 - Y.T) * np.log(1 - y_pred_clipped))
    gradients = {}
    
    # Output layer gradient
    output_idx = len(layer_dims) - 2
    gradients[f'dz{output_idx}'] = y_pred_clipped - Y.T
    gradients[f'dW{output_idx}'] = (1 / X.shape[0]) * np.dot(gradients[f'dz{output_idx}'], cache[output_idx - 1][0].T)
    gradients[f'dB{output_idx}'] = (1 / X.shape[0]) * np.sum(gradients[f'dz{output_idx}'], axis=1, keepdims=True)
    
    # Hidden layers gradients
    for i in range(output_idx - 1, -1, -1):
        gradients[f'dz{i}'] = np.dot(W[i + 1].T, gradients[f'dz{i + 1}']) * (cache[i][1] > 0)
        if i > 0:
            gradients[f'dW{i}'] = (1 / X.shape[0]) * np.dot(gradients[f'dz{i}'], cache[i - 1][0].T)
        else:
            gradients[f'dW{i}'] = (1 / X.shape[0]) * np.dot(gradients[f'dz{i}'], X)
        gradients[f'dB{i}'] = (1 / X.shape[0]) * np.sum(gradients[f'dz{i}'], axis=1, keepdims=True)
    
    # Update weights and biases
    for i in range(len(layer_dims) - 1):
        Vdw[i] = B1 * Vdw[i] + (1 - B1) * gradients[f'dW{i}']
        Sdw[i] = B2 * Sdw[i] + (1 - B2) * (gradients[f'dW{i}'] ** 2)
        Vdb[i] = B1 * Vdb[i] + (1 - B1) * gradients[f'dB{i}']
        Sdb[i] = B2 * Sdb[i] + (1 - B2) * (gradients[f'dB{i}'] ** 2)
        W[i] -= learning_rate * (Vdw[i] / np.sqrt(Sdw[i] + epsilon))
        B[i] -= learning_rate * (Vdb[i] / np.sqrt(Sdb[i] + epsilon))
    return W, B, cost


Train the deep network with mini-batches and Adam-style updates over multiple epochs. Mini-batches make each gradient cheaper and noisier than a full-dataset gradient; shuffling prevents the model from seeing examples in the same order every epoch.

The moment arrays `Vdw`, `Sdw`, `Vdb`, and `Sdb` persist across batches, which is what lets Adam remember recent gradient direction and magnitude instead of treating every batch independently.


In [ ]:
def train_model(X, Y, layer_dims, learning_rate = 0.01, epoch = 500, batch_size = 32):
    W, B = initialize_parameters(layer_dims)
    Vdw = [0 for l in range(len(layer_dims) - 1)]; Sdw = [0 for l in range(len(layer_dims) - 1)]; Vdb = [0 for l in range(len(layer_dims) - 1)]; Sdb = [0 for l in range(len(layer_dims) - 1)]
    number_of_batches = int(np.ceil(X.shape[0] / batch_size))
    for i in range(epoch):
        perm = np.random.permutation(X.shape[0])
        X_shuffled = X[perm]
        Y_shuffled = Y[perm]
        for j in range(number_of_batches):
            X_batch = X_shuffled[j * batch_size:(j + 1) * batch_size]
            Y_batch = Y_shuffled[j * batch_size:(j + 1) * batch_size]
            W, B, cost = gradient_descent(X_batch, Y_batch, W, B, Vdw, Sdw, Vdb, Sdb,layer_dims, learning_rate)
            print(f"Batch {j} - Cost for epoch {i} is {cost}")
    return W, B


Convert output probabilities into binary labels with a 0.5 decision threshold. This turns the smooth sigmoid score used for learning into the hard class decision used for accuracy.


In [ ]:
def predict(pred):
    return (pred > 0.5).astype(int)

Train the network with mini-batch Adam-style updates, then generate predictions for the test set. The printed shapes confirm that the input matrix and labels still match after batching and reshaping.


In [ ]:
print(X_flat.shape, y_train.shape)
W, B = train_model(X_flat, y_train, layer_dims, epoch=5000)
pred = forward_pass(X_test, W, B, layer_dims)[0]

Compute accuracy and visualize one test prediction from the trained optimizer run. Train accuracy measures fit to examples used for updates; test accuracy estimates whether the learned features transfer to held-out images.


In [ ]:
import matplotlib.pyplot as plt

train_acc = (predict(forward_pass(X_flat, W, B, layer_dims)[0]) == y_train).mean()
test_acc = (predict(forward_pass(X_test, W, B, layer_dims)[0]) == y_test).mean()
print(f"train acc: {train_acc:.3f}   test acc: {test_acc:.3f}")

test_idx = 8
test_image = X_test[test_idx].reshape(1, -1)
prediction_prob = forward_pass(test_image, W, B, layer_dims)[0]
prediction = predict(prediction_prob)

img_display = (X_test[test_idx].reshape(128, 128, 3) * 255).astype(np.uint8)
true_label = int(y_test[test_idx][0])

plt.figure(figsize=(6, 6))
plt.imshow(img_display)
plt.title(f"Pred: {'Gopher' if prediction[0][0] == 1 else 'Not Gopher'} ({prediction_prob[0][0]:.4f}) | True: {'Gopher' if true_label == 1 else 'Not Gopher'}")
plt.axis('off')
plt.show()

Repeat the prediction visualization for the first test image. The probability is useful because two correct labels can still have very different confidence levels.


In [ ]:
import matplotlib.pyplot as plt

train_acc = (predict(forward_pass(X_flat, W, B, layer_dims)[0]) == y_train).mean()
test_acc = (predict(forward_pass(X_test, W, B, layer_dims)[0]) == y_test).mean()
print(f"train acc: {train_acc:.3f}   test acc: {test_acc:.3f}")

test_idx = 0
test_image = X_test[test_idx].reshape(1, -1)
prediction_prob = forward_pass(test_image, W, B, layer_dims)[0]
prediction = predict(prediction_prob)

img_display = (X_test[test_idx].reshape(128, 128, 3) * 255).astype(np.uint8)
true_label = int(y_test[test_idx][0])

plt.figure(figsize=(6, 6))
plt.imshow(img_display)
plt.title(f"Pred: {'Gopher' if prediction[0][0] == 1 else 'Not Gopher'} ({prediction_prob[0][0]:.4f}) | True: {'Gopher' if true_label == 1 else 'Not Gopher'}")
plt.axis('off')
plt.show()

Show a grid of test images with true labels so the model inputs can be inspected visually. This keeps the optimizer results connected to the actual visual classification task.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Show a grid of test images with true labels.
fig, axes = plt.subplots(2, 6, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    img = (X_test[i].reshape(128, 128, 3) * 255).astype(np.uint8)
    ax.imshow(img)
    ax.set_title(f"y={int(y_test[i][0])}")
    ax.axis('off')
plt.tight_layout()
plt.show()

Reload and display raw examples from both classes. Side-by-side examples help explain why optimization matters: the model is trying to separate visually similar categories from limited pixels.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("data/gopher_dataset.npz")
Xtr_raw, ytr_raw = d["X_train"], d["y_train"]
Xte_raw, yte_raw = d["X_test"], d["y_test"]
print("shapes:", Xtr_raw.shape, ytr_raw.shape, Xte_raw.shape, yte_raw.shape)

# Show 6 gophers and 6 non-gophers.
gophers = Xtr_raw[ytr_raw == 1][:6]
nongophers = Xtr_raw[ytr_raw == 0][:6]

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    axes[0, i].imshow(gophers[i]); axes[0, i].axis("off"); axes[0, i].set_title("gopher")
    axes[1, i].imshow(nongophers[i]); axes[1, i].axis("off"); axes[1, i].set_title("not gopher")
plt.tight_layout(); plt.show()